# Quickstart — `cascade_av`

This notebook introduces the CASCADE DevKit. You will
load the corpus, inspect a single clip's annotation, and run
your first queries in the embedded DSL.


## About the dataset

The **CASCADE dataset** adds causal and spatio-temporal action
annotations on top of NVIDIA's *Physical AI AV Dataset*. Each
clip is a short front-facing driving video paired with a JSON
annotation bundle (schema `2.0.0`). The DevKit parses those
bundles into a typed Pydantic tree and exposes a small query
language for searching the corpus.

### Annotation scope

Every clip is annotated with the following entity categories:

| entity | what it captures |
|---|---|
| **Agents** | non-ego actors — vehicles, pedestrians, cyclists, animals, officers — with type, ego-relative position, visibility intervals, and per-interval actions |
| **Ego vehicle** | actions taken by the recording car (drive, decel, stop, turn, change-lane, …) and the entities that motivated them |
| **Environments** | road, intersection, crosswalk, sidewalk, roundabout, cycle-lane, tunnel, … with lane counts and one-way flags |
| **Conditions** | weather, lighting, construction, occlusion |
| **Traffic lights** | signal-head colors and state transitions, plus annotator-tagged flags such as `could_have_cleared` and `ego_in_on_yellow` |
| **Traffic objects** | stop signs, yield signs, cones, debris, barriers |

### What makes it *causal*

The schema includes a `because_of` edge on every action: each
action can name the entities that caused it. "Ego decelerates
*because of* a pedestrian" is not reconstructed from
co-occurrence — it is annotated explicitly. The query DSL
surfaces this directly through the `because_of` operator.

### Schema at a glance

Each clip's annotation is a single JSON file with four
top-level keys:

- `video` — clip metadata (fps, duration, `clip_id`).
- `annotation` — typed entity arrays
  (`agents[]`, `ego_vehicle`, `environments[]`, `conditions[]`,
  `traffic_lights[]`, `traffic_objects[]`).
- `provenance` — annotator metadata.
- `schema_version` — currently `2.0.0`.

The corpus shipped with this DevKit contains 376 annotation
files covering roughly 300 distinct clips.

### What this notebook covers

1. Load the corpus and inspect a single clip's annotation.
2. Run your first DSL queries against the whole corpus.
3. Read back the `MatchSet` result type.
4. Scope the same DSL to a single clip.

> Set the `CASCADE_AV_DATASET_ROOT` environment variable to the
> directory containing the JSON annotations before running the
> setup cell below.


## Setup


In [1]:
import os
from pathlib import Path

from cascade_av.dataset import CascadeDataset
from cascade_av.io import load_dir
from cascade_av.query import find_on_bundle

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [2]:
dataset_root = Path(os.environ["CASCADE_AV_DATASET_ROOT"])
ds = CascadeDataset(dataset_root)
print(f"corpus loaded — {len(ds)} clips")


corpus loaded — 344 clips


/home/horde/Projects/av-causal-dataset-tools-feat-dsl-attribute-surface/src/cascade_av/dataset.py:118: UserWarning: 1 clip_id(s) had multiple annotation files; kept first by filename. (Set CASCADE_AV_VERBOSE=1 for per-file detail.)
  self._scan(annotation_paths, annotation_root)


## 1. Inspect one clip's annotation

`CascadeDataset.get_sequence` downloads the clip video lazily by
default. To keep this tour offline-friendly we parse the JSON
annotations directly via the local `io.load_dir` helper.


In [3]:
bundles = load_dir(dataset_root)
b = bundles[0]

summary = pd.DataFrame([
    ("clip_id",        b.video.clip_id),
    ("schema_version", b.schema_version),
    ("fps",            b.video.fps),
    ("duration_s",     b.video.duration_s),
    ("agents",         len(b.annotation.agents)),
    ("ego actions",    len(b.annotation.ego_vehicle.actions)),
    ("environments",   len(b.annotation.environments)),
    ("traffic lights", len(b.annotation.traffic_lights)),
    ("traffic objects", len(b.annotation.traffic_objects)),
], columns=["field", "value"])
summary


,field,value
0,clip_id,042ec95b-c21e-4b2c-a715-a7fb5e7c0f38
1,schema_version,2.0.0
2,fps,30.00
3,duration_s,20.13
4,agents,1
5,ego actions,4
6,environments,4
7,traffic lights,0
8,traffic objects,11


## 2. Run a DSL query against the whole corpus

`CascadeDataset.count(query)` returns the number of clips with
at least one match. Queries are plain strings in the DSL — the
full grammar is covered in `02_query_dsl_tour.ipynb`.


In [4]:
queries = [
    "agent.type = ped",
    "agent.type = vehicle",
    "agent.type = ped and env.type = crosswalk",
    "ego.action = stop and light.color = red",
]
pd.DataFrame(
    [(q, ds.count(q)) for q in queries],
    columns=["query", "matching clips"],
)


,query,matching clips
0,agent.type = ped,133
1,agent.type = vehicle,313
2,agent.type = ped and env.type = crosswalk,92
3,ego.action = stop and light.color = red,52


## 3. `find()` returns the full `MatchSet`

Every match is a `(clip_id, entity, interval)` tuple. The
`.clips()`, `.entities()`, and `.intervals()` projections pull
each column individually.


In [5]:
matches = ds.find("agent.type = ped and env.type = crosswalk")
clips = sorted(set(matches.clips()))

print(f"{len(matches)} matches across {len(clips)} distinct clips")
pd.DataFrame({"clip_id": clips[:5]})


388 matches across 92 distinct clips


,clip_id
0,02a0d178-4f95-438f-b45a-c0f65d0d765a
1,038b067d-71fc-4c1c-b634-8985a82290bb
2,038f960a-9b8c-4337-81ef-0ee8461b490d
3,050ae8d3-41ab-4f66-b5b0-0e7bf21d31ad
4,0683f871-7d49-4bf9-8201-6227af1221bc


## 4. The same DSL on a single clip

`find_on_bundle` runs a query against one parsed bundle and
returns a `MatchSet` scoped to that clip only — useful when you
already have a bundle in hand.


In [6]:
single = find_on_bundle(b, "agent.type = vehicle")
print(f"{len(single)} vehicle matches in clip {b.video.clip_id}")
pd.DataFrame({"entity": [str(e) for e in single.entities()[:5]]})


1 vehicle matches in clip 042ec95b-c21e-4b2c-a715-a7fb5e7c0f38


,entity
0,id='Agent1' name='' amount='Single' type='oxd:Car' other_type_description='' visibility_start_timestamp='0...


## 5. What else was happening?

Each `Match` carries `(clip_id, entity, interval)`. Given that
interval, `ds.context_for(match)` returns a `ContextWindow`
snapshot of every entity in the clip whose annotated time
range overlaps the match: visible agents (with the actions
they were doing at the time), ego actions, environments,
conditions, traffic-light states, and persistent traffic
objects.

Temporal operators like `while` tighten each match's interval
to the intersection, so the context window is the actual
moment both conditions held.


In [7]:
matches = ds.find("agent.type = ped while ego.action = decel")
m = matches.matches[0]
ctx = ds.context_for(m)

pd.DataFrame([
    ("clip_id",        ctx.clip_id),
    ("window",         f"{m.interval.start:.2f}–{m.interval.end:.2f}s"),
    ("agents",         len(ctx.agents)),
    ("ego actions",    [ea.type for ea in ctx.ego_actions]),
    ("environments",   [e.type for e in ctx.environments]),
    ("light states",   len(ctx.light_states)),
    ("traffic objects", [o.type for o in ctx.traffic_objects]),
], columns=["field", "value"])


,field,value
0,clip_id,a9167a89-a730-437a-b07f-893b3949fefc
1,window,13.00–13.30s
2,agents,4
3,ego actions,"[oxd:Stop, oxd:Decelerate]"
4,environments,"[oxd:Road, oxd:Sidewalk, oxd:PedestrianCrossing, oxd:YIntersection]"
5,light states,0
6,traffic objects,"[Other, Other oxd:TrafficSign, oxd:WarningSign]"


Drill into the agents — each `AgentInWindow` carries the agent
itself, its visibility window clipped to the match, and the
actions of that agent that were active in the window.


In [8]:
pd.DataFrame([
    {
        "id":         a.agent.id,
        "type":       a.agent.type,
        "visible":    f"{a.visibility.start:.2f}–{a.visibility.end:.2f}s",
        "actions":    [ax.action_type for ax in a.actions],
    }
    for a in ctx.agents
])


,id,type,visible,actions
0,Agent3,Pedestrian (Adult),13.00–13.30s,[oxd:Walk]
1,Agent4,Pedestrian (Adult),13.00–13.30s,[oxd:Walk]
2,Agent5,Pedestrian (Adult),13.00–13.30s,[oxd:Walk]
3,Agent6,oxd:Car,13.00–13.30s,[oxd:MakeARightTurn (unprotected)]
